# 🏨 Hotel do Mosquito — Acesso via Navegador

**Laboratório de Banco de Dados**  
Camili de Moura Marangoni · Lucas Sobrinho Santos · Maria Eduarda Patu Ângelo da Silva  
Matheus Pinheiro de Camargo Silva · Willian Alexandre Schwingel Ferreira

---

## Como usar

Execute as **3 células abaixo em ordem**. Clique em ▶ e **aguarde o ✅** antes de passar para a próxima.

| Célula | O que faz | Tempo estimado |
|---|---|---|
| **Célula 1** | Instala MySQL, display virtual, noVNC e clona o projeto | ~2 minutos |
| **Célula 2** | Carrega o banco de dados (tabelas, procedures, dados) | ~15 segundos |
| **Célula 3** | Abre o sistema e exibe o link para acessar no navegador | ~30 segundos |

---

### Credenciais

| Login | Senha | Perfil |
|---|---|---|
| gerente1 | senha123 | Gerente |
| gerente2 | senha123 | Gerente |
| recep1 | senha123 | Recepcionista |
| recep2 | senha123 | Recepcionista |
| recep3 | senha123 | Recepcionista |

In [ ]:
# ══════════════════════════════════════════════════════════════════
# CÉLULA 1 — Instalar e configurar o ambiente (~2 minutos)
# Aguarde o ✅ AMBIENTE PRONTO ao final antes de continuar
# ══════════════════════════════════════════════════════════════════
import os, sys, subprocess, configparser, time
from pathlib import Path

def sh(cmd):
    return subprocess.run(cmd, shell=True, capture_output=True, text=True)

def ok(msg):  print(f'  ✅ {msg}')
def run(msg, cmd):
    print(f'  ⏳ {msg}...')
    sh(cmd)

print('📦 PASSO 1/5 — Instalando pacotes do sistema...')
run('MySQL Server',   'apt-get install -y mysql-server > /dev/null 2>&1')
run('Xvfb + x11vnc',  'apt-get install -y xvfb x11vnc websockify > /dev/null 2>&1')
run('noVNC',          'git clone --depth=1 https://github.com/novnc/noVNC.git /opt/noVNC > /dev/null 2>&1 || true')
run('cloudflared',    'wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared')
ok('Pacotes instalados')

print('\n🔐 PASSO 2/5 — Configurando MySQL...')
sh('service mysql start')
time.sleep(2)
sh("mysql -e \"ALTER USER 'root'@'localhost' IDENTIFIED WITH mysql_native_password BY 'hotel123'; FLUSH PRIVILEGES;\"")
ok('MySQL pronto (root / hotel123)')

print('\n📂 PASSO 3/5 — Clonando repositório...')
if not Path('hotel-mosquito-labdb').exists():
    run('Clonando', 'git clone https://github.com/Matheus-PC-Silva/hotel-mosquito-labdb.git -q')
else:
    run('Atualizando', 'git -C hotel-mosquito-labdb pull -q')
ok('Repositório pronto')

print('\n📦 PASSO 4/5 — Instalando dependências Python...')
run('mysql-connector + tkcalendar',
    f'{sys.executable} -m pip install mysql-connector-python==8.4.0 tkcalendar==1.6.1 -q')
ok('Pacotes Python instalados')

print('\n⚙️  PASSO 5/5 — Configurando conexão...')
cfg_path = Path('hotel-mosquito-labdb/app/config.ini')
cfg = configparser.ConfigParser()
cfg.read(cfg_path)
cfg['database']['port']     = '3306'
cfg['database']['host']     = 'localhost'
cfg['database']['password'] = 'hotel123'
with open(cfg_path, 'w') as f:
    cfg.write(f)
proj = str(Path('hotel-mosquito-labdb').resolve())
if proj not in sys.path:
    sys.path.insert(0, proj)
ok('config.ini ajustado para MySQL local (porta 3306)')

print('\n✅ AMBIENTE PRONTO — Execute a Célula 2')

In [ ]:
# ══════════════════════════════════════════════════════════════════
# CÉLULA 2 — Carregar banco de dados (~15 segundos)
# ══════════════════════════════════════════════════════════════════
import subprocess
from pathlib import Path

sql_file = Path('hotel-mosquito-labdb/sql/hotel_mosquito_full.sql')
print('🗄️  Criando schema, views, procedures, triggers e dados...')

result = subprocess.run(
    ['mysql', '-uroot', '-photel123', '--default-character-set=utf8mb4'],
    input=sql_file.read_bytes(),
    capture_output=True
)

if result.returncode != 0:
    print('❌ Erro ao carregar SQL:')
    print(result.stderr.decode(errors='replace'))
    raise SystemExit('Verifique o erro acima antes de continuar.')

check = subprocess.run(
    ['mysql', '-uroot', '-photel123', 'hotel_mosquito', '-t', '-e',
     'SELECT '
     '(SELECT COUNT(*) FROM Categoria_Quarto) AS categorias,'
     '(SELECT COUNT(*) FROM Quarto)           AS quartos,'
     '(SELECT COUNT(*) FROM Cliente)          AS clientes,'
     '(SELECT COUNT(*) FROM Funcionario)      AS funcionarios,'
     '(SELECT COUNT(*) FROM Produto_Servico)  AS produtos,'
     '(SELECT COUNT(*) FROM Reserva)          AS reservas,'
     '(SELECT COUNT(*) FROM Hospedagem)       AS hospedagens,'
     '(SELECT COUNT(*) FROM Consumo)          AS consumos;'],
    capture_output=True, text=True
)
print('\n📊 Dados carregados (esperado: 5 | 15 | 10 | 5 | 10 | 10 | 8 | 15):')
print(check.stdout)

procs = subprocess.run(
    ['mysql', '-uroot', '-photel123', 'hotel_mosquito', '-N', '-e',
     'SELECT COUNT(*) FROM information_schema.ROUTINES '
     'WHERE ROUTINE_SCHEMA="hotel_mosquito" AND ROUTINE_TYPE="PROCEDURE";'],
    capture_output=True, text=True
)
print(f'📋 Stored Procedures: {procs.stdout.strip()} (esperado: ~26)')
print('\n✅ BANCO PRONTO — Execute a Célula 3')

In [ ]:
# ══════════════════════════════════════════════════════════════════
# CÉLULA 3 — Abrir o sistema no navegador
# Aguarde o link 👉 aparecer, depois abra-o no navegador
# ══════════════════════════════════════════════════════════════════
import subprocess, os, re, time
from pathlib import Path

proj_dir = str(Path('hotel-mosquito-labdb').resolve())

# 1. Display virtual
print('🖥️  Iniciando display virtual...')
subprocess.Popen(
    ['Xvfb', ':99', '-screen', '0', '1280x800x24', '-ac'],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
)
time.sleep(1.5)
os.environ['DISPLAY'] = ':99'
print('  ✅ Display :99 ativo')

# 2. Servidor VNC
print('📡 Iniciando VNC server...')
subprocess.Popen(
    ['x11vnc', '-display', ':99', '-forever', '-nopw', '-quiet', '-xkb'],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
)
time.sleep(2)
print('  ✅ VNC pronto (porta 5900)')

# 3. Proxy noVNC (VNC → WebSocket → browser)
print('🌐 Iniciando noVNC...')
subprocess.Popen(
    ['python3', '-m', 'websockify', '--web=/opt/noVNC', '6080', 'localhost:5900'],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
)
time.sleep(2)
print('  ✅ noVNC pronto (porta 6080)')

# 4. Túnel público via Cloudflare (sem cadastro, sem token)
print('🔗 Criando túnel público (Cloudflare)...')
cf = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://localhost:6080'],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.PIPE,
    text=True
)

url = None
for _ in range(40):           # aguarda até ~40s pelo URL
    line = cf.stderr.readline()
    m = re.search(r'https://[\w\-]+\.trycloudflare\.com', line)
    if m:
        url = m.group(0)
        break

if not url:
    print('  ⚠️  Cloudflare não retornou URL. Tente executar esta célula novamente.')
    url = 'http://localhost:6080'
else:
    print('  ✅ Túnel criado (sem login necessário)')

# 5. Iniciar a aplicação
print('\n🚀 Iniciando Hotel do Mosquito...')
subprocess.Popen(
    ['python3', '-m', 'app.main'],
    cwd=proj_dir,
    env={**os.environ, 'DISPLAY': ':99'},
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
)
time.sleep(3)

# 6. Exibir link
link = f"{url}/vnc.html?autoconnect=true&resize=scale&quality=7"
print()
print('═' * 62)
print('  🎉 SISTEMA PRONTO!')
print('═' * 62)
print()
print('  Abra este link no seu navegador (celular ou computador):')
print()
print(f'  👉  {link}')
print()
print('  Credenciais:')
print('  ┌──────────┬──────────┬────────────────┐')
print('  │ gerente1 │ senha123 │ Gerente        │')
print('  │ gerente2 │ senha123 │ Gerente        │')
print('  │ recep1   │ senha123 │ Recepcionista  │')
print('  └──────────┴──────────┴────────────────┘')
print()
print('  ⚠️  Mantenha esta aba do Colab aberta — fechar encerra o sistema.')
print('═' * 62)

---
## ℹ️ Informações adicionais

### O link não apareceu ou apareceu `localhost:6080`?
Execute a **Célula 3** novamente — o túnel Cloudflare pode demorar alguns segundos extras.

### O Colab desconectou?
Execute as **Células 1, 2 e 3** em ordem novamente.

### A tela está pequena no celular?
`resize=scale` ajusta automaticamente. Use pinça para zoom extra.

### Dicas de uso no celular
- **Toque simples** → clique do mouse
- **Toque longo** → clique direito
- **Teclado** → aparece automaticamente ao tocar em campos de texto

### Por que Cloudflare e não ngrok?
O Cloudflare Tunnel (`trycloudflare.com`) é gratuito e não exige cadastro nem token.  
O ngrok passou a exigir autenticação mesmo para uso básico.

---
*Hotel do Mosquito — LABDB · https://github.com/Matheus-PC-Silva/hotel-mosquito-labdb*